In [1]:
!pip install -U ultralytics
import ultralytics
ultralytics.checks()

Ultralytics 8.4.115 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition, 97250MiB)
Setup complete ✅ (48 CPUs, 176.9 GB RAM, 46.7/112.6 GB disk)


In [2]:
!pip install roboflow

from roboflow import Roboflow
import os
rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])
project = rf.workspace("egitim-i2v3l").project("celik_kubbe-3tsml")
version = project.version(7)
dataset = version.download("yolo26")
                

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 285.0/285.0 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 66.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 125.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 179.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 8.8 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 5.0.0.93
    Uninstalling opencv-python-headless-5.0.0.93:
      Successfully uninstalled opencv-python-headless-5.0.0.93
  Attempting uninstall: typer
    Found existing installation: typer 0.26.8
    Uninstalling typer-0.26.8:
      Successfully uninstalled typer-0.26.8


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Celik_Kubbe-7 in yolo26:: 100%|██████████| 6933/6933 [00:00<00:00, 29221.74it/s]


In [4]:
dataset.location

'/content/Celik_Kubbe-7'

In [ ]:
"""
COLAB EGITIM SCRIPTI
====================
Hedef : drone / f16 / rocket / helicopter ayrimi
Ortam : kapali alan, degisken isik, gimbal (pan-tilt) kamera
Deploy: Hailo-8 (INT8, sabit input shape, batch=1)

argparse YOK -> Colab/Jupyter'da dogrudan calisir.
Ayarlari asagidaki CFG sozlugunden degistir.

Colab'da 3 hucre halinde kullan:
  Hucre 1: !pip install ultralytics albumentations
  Hucre 2: (bu dosyanin tamami)
  Hucre 3: main()
"""

from pathlib import Path

from ultralytics import YOLO

# ==========================================================================
# 0) VERI SETI YOLU  <-- BURAYI DOLDUR
# ==========================================================================
# Roboflow kullaniyorsan:
#     from roboflow import Roboflow
#     rf = Roboflow(api_key="...")
#     dataset = rf.workspace("...").project("...").version(1).download("yolov11")
#     DATA_YAML = dataset.location + "/data.yaml"     # <-- ".location" TEK BASINA
#                                                     #     klasordur, YAML degil!
#
# Google Drive'dan:
#     from google.colab import drive; drive.mount("/content/drive")
#     DATA_YAML = "/content/drive/MyDrive/dataset/data.yaml"
#
# Manuel yukleme:
#     DATA_YAML = "/content/dataset/data.yaml"

DATA_YAML = "/content/dataset/data.yaml"


def veri_yolu_bul():
    """Roboflow 'dataset' degiskeni tanimliysa otomatik yakalar."""
    d = globals().get("dataset", None)
    if d is not None and hasattr(d, "location"):
        p = Path(d.location) / "data.yaml"
        if p.exists():
            print(f"[BILGI] Roboflow veri seti bulundu: {p}")
            return str(p)
    return DATA_YAML


# ==========================================================================
# 1) AYARLAR
# ==========================================================================

CFG = dict(
    model="yolo11m.pt",     # yolo11m (dogruluk) | yolo11s (hiz)
    #                         YOLO26 KULLANMA: Hailo-8'de INT8 kaybi daha yuksek
    name="hava_araci_v1",
    device=None,            # None = otomatik. Colab'da GPU varsa 0 secer.
)

HYP = dict(
    # ---- Egitim suresi ----
    epochs=300,
    patience=80,

    # ---- Cozunurluk ve batch ----
    imgsz=640,
    batch=16,               # SABIT DEGER ONERILIR.
    #                         batch=0.80 da gecerli (GPU belleginin %80'i) ama
    #                         Colab T4'te (16GB) yolo11m@640 icin ~10-12'ye duser.
    #                         BatchNorm istatistikleri 16 altinda bozulmaya baslar.
    #                         OOM alirsan: 8'e in, daha asagi INME -> model yerine
    #                         yolo11s.pt'ye gec.

    # ---- Optimizasyon ----
    optimizer="auto",
    lr0=0.01,
    lrf=0.01,
    cos_lr=True,
    warmup_epochs=3.0,
    weight_decay=0.0005,
    amp=True,

    # ---- KAYIP AGIRLIKLARI: sinif ayrimi 1. oncelik ----
    cls=0.9,                # varsayilan 0.5 -> 0.9
    box=7.5,
    dfl=1.5,
    cls_pw=0.25,            # sinif dengesizligi. Nadir sinif zayifsa 1.0 yap.

    # ---- GEOMETRIK AUGMENTASYON ----
    degrees=25.0,           # gimbal egimi + maket yonelimleri
    translate=0.15,
    scale=0.7,
    shear=2.0,
    perspective=0.0005,
    fliplr=0.5,
    flipud=0.2,

    # ---- RENK: renk kisayolunu kirmak icin agresif ----
    hsv_h=0.05,             # varsayilan 0.015. Model rengi degil SEKLI ogrensin.
    hsv_s=0.8,
    hsv_v=0.6,              # degisken ic mekan aydinlatmasi

    # ---- KOMPOZIT ----
    mosaic=0.8,
    close_mosaic=25,
    mixup=0.0,              # ACMA: ince taneli sinif ayrimina zarar verir
    cutmix=0.05,
    multi_scale=0.3,

    # ---- Diger ----
    plots=True,
    val=True,
    save_period=25,         # Colab kopmasina karsi sigorta
    seed=0,
    deterministic=True,
    workers=2,              # Colab'da 8 degil 2 -> fazlasi RAM sisirir/asilir
)


# ==========================================================================
# 2) KAPALI ORTAM + GIMBAL ICIN OZEL ALBUMENTATIONS
# ==========================================================================

def build_augmentations():
    try:
        import albumentations as A
    except ImportError:
        print("[UYARI] albumentations yok -> ozel augmentasyon atlaniyor.")
        print("        !pip install albumentations")
        return None

    candidates = [
        (A.MotionBlur, dict(blur_limit=(3, 15), p=0.50)),      # gimbal + dusuk isik
        (A.Defocus, dict(radius=(1, 4), p=0.15)),
        (A.RandomBrightnessContrast, dict(brightness_limit=0.35, contrast_limit=0.30, p=0.50)),
        (A.RandomGamma, dict(gamma_limit=(70, 140), p=0.30)),
        (A.CLAHE, dict(clip_limit=3.0, p=0.20)),
        (A.ColorJitter, dict(brightness=0.0, contrast=0.0, saturation=0.4, hue=0.05, p=0.30)),
        (A.ISONoise, dict(p=0.30)),
        (A.ImageCompression, dict(quality_range=(40, 90), p=0.30)),
    ]

    transforms = []
    for cls, kwargs in candidates:
        try:
            transforms.append(cls(**kwargs))
        except Exception as e:
            try:
                transforms.append(cls(p=kwargs.get("p", 0.3)))
                print(f"[BILGI] {cls.__name__} varsayilan parametrelerle eklendi ({e}).")
            except Exception:
                print(f"[UYARI] {cls.__name__} atlandi (surum uyumsuz).")

    print(f"[BILGI] {len(transforms)} ozel Albumentations transformu aktif.")
    return transforms


# ==========================================================================
# 3) ANA AKIS
# ==========================================================================

def main(data=None, model=None, imgsz=None, epochs=None, batch=None,
         name=None, resume=False, use_albumentations=True):
    """
    Ornek kullanim:
        main()
        main(model="yolo11s.pt", epochs=150)
        main(imgsz=768, batch=8)
        main(resume=True)
    """
    data = data or veri_yolu_bul()
    cfg = {**CFG, **{k: v for k, v in
                     dict(model=model, name=name).items() if v is not None}}
    hyp = dict(HYP)
    for k, v in dict(imgsz=imgsz, epochs=epochs, batch=batch).items():
        if v is not None:
            hyp[k] = v
    if cfg["device"] is not None:
        hyp["device"] = cfg["device"]

    if not Path(data).exists():
        raise SystemExit(
            f"[HATA] Veri seti YAML bulunamadi: {data}\n"
            f"       DATA_YAML degiskenini duzelt.\n"
            f"       Roboflow ise: dataset.location + '/data.yaml'"
        )

    if resume:
        m = YOLO(f"runs/detect/{cfg['name']}/weights/last.pt")
        return m.train(resume=True)

    m = YOLO(cfg["model"])

    if use_albumentations:
        augs = build_augmentations()
        if augs:
            hyp["augmentations"] = augs

    print("=" * 70)
    print(f"Model  : {cfg['model']}")
    print(f"Veri   : {data}")
    print(f"imgsz  : {hyp['imgsz']}   (Hailo HEF ayni deger ile derlenecek!)")
    print(f"batch  : {hyp['batch']}")
    print(f"epochs : {hyp['epochs']}")
    print("=" * 70)

    results = m.train(
        data=data,
        project="runs/detect",
        name=cfg["name"],
        exist_ok=True,       # Colab'da tekrar calistirinca hata vermesin
        **hyp,
    )

    d = f"runs/detect/{cfg['name']}"
    print("\n" + "=" * 70)
    print("EGITIM BITTI")
    print("=" * 70)
    print(f"Agirliklar : {d}/weights/best.pt")
    print(f"ONCE BUNA BAK: {d}/confusion_matrix_normalized.png")
    print("   -> Asil metrigin bu. 'helikopter kac kere drone sanildi'.")
    print(f"Etiket dagilimi: {d}/labels.jpg")
    print("   -> Cok sayida kutu <20px ise imgsz'i 768'e cikar.")
    print(f"Ilk epoch mozaikleri: {d}/train_batch0.jpg")
    print("   -> Etiketlerin dogru oldugunu GOZLE dogrula.")
    print("\nHAILO EXPORT (egitimdekiyle AYNI imgsz):")
    print(f'''
    m = YOLO("{d}/weights/best.pt")
    m.export(format="hailo", name="hailo8",
             imgsz={hyp["imgsz"]},
             data="{data}",     # kalibrasyon: KENDI verinle, >=1024 kare
             conf=0.15,          # NMS esigi HEF'e gomulur, sonra degistiremezsin
             iou=0.7)
    ''')
    print("Not: Derleme Linux x86_64 + Hailo DFC v3.x gerektirir (Colab'da olmaz).")
    return results


# Colab'da otomatik baslamasin diye __main__ blogu YOK.
# Egitimi baslatmak icin ayri bir hucrede:  main()

usage: colab_kernel_launcher.py [-h] [--data DATA] [--model MODEL]
                                [--imgsz IMGSZ] [--epochs EPOCHS]
                                [--batch BATCH] [--device DEVICE]
                                [--name NAME] [--resume] [--no-albu]
colab_kernel_launcher.py: error: unrecognized arguments: -f /root/.local/share/jupyter/runtime/kernel-032cdd09-dba2-434e-aad6-7c066b5507ee.json


SystemExit: 2